<a href="https://colab.research.google.com/github/taylan-sen/CIS355_FALL2026/blob/main/AStar_Search_Maze_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From Dijkstra to A\*: Adding a Heuristic to Shortest-Path Search

**Goal of this notebook:** you already know Dijkstra's algorithm. Here you will see that **A\* is Dijkstra with one extra term** — a *heuristic* estimate of the remaining distance — and you will watch how that one term changes which cells of a maze get explored.

By the end you should be able to:

1. Explain the priority used by Dijkstra ($g$) and by A\* ($f = g + h$).
2. Implement Manhattan and Euclidean heuristics on a grid.
3. Explain **admissibility** and **consistency**, and why they guarantee an optimal path.
4. Predict what happens when the heuristic is too weak (→ Dijkstra), too strong (→ fast but wrong), or ignores $g$ entirely (→ Greedy Best-First).

**How to use it:** run the cells top to bottom (`Runtime → Run all` works). Everything uses only `numpy`, `matplotlib`, `pandas` and `ipywidgets`, all preinstalled in Google Colab.

**Color legend used in every picture**

| Color | Meaning |
|---|---|
| dark gray | wall |
| blue shades | cells **expanded** (popped from the priority queue); lighter = earlier, darker = later |
| yellow | cells on the **frontier** (in the priority queue, not yet expanded) |
| red line | the path that was returned |
| green circle / red star | start / goal |

In [ ]:
# Setup -- run this first
import heapq, math, itertools, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

plt.rcParams["figure.dpi"] = 100
plt.rcParams["animation.embed_limit"] = 60  # MB, lets longer animations render inline
print("Ready.")

## Part 1 — A maze is just a graph

We draw a maze as text: `#` is a wall, `.` is open floor, `S` is the start and `G` is the goal.

Each open cell is a **node**. Two open cells that share a side are connected by an **edge** of cost 1 (we call this a *4-connected* grid: up, down, left, right). Later we will also allow diagonal moves of cost $\sqrt{2}$.

In [ ]:
MAZE_TEXT = '''
##############################
#S.....#.........#...........#
#.####.#.#######.#.#########.#
#.#....#.#.....#.#.#.......#.#
#.#.####.#.###.#...#.#####.#.#
#.#......#...#.#####.#...#...#
#.##########.#.......#.#.#####
#............#########.#.....#
#####.######...........#####.#
#.....#....#.#########.#.....#
#.#####.##.#.#.......#.#.#####
#.......#..#...#####.#.#.....#
#.#######.######...#.#.#####.#
#.........#........#.......#G#
##############################
'''

OPEN_TEXT = '''
##############################
#............................#
#............................#
#..........#######...........#
#................#...........#
#................#...........#
#.....S..........#........G..#
#................#...........#
#................#...........#
#..........#######...........#
#............................#
#............................#
##############################
'''

def parse_grid(text):
    # Return (grid, start, goal). grid[r, c] == 1 means wall.
    rows = [line for line in text.strip("\n").splitlines() if line.strip()]
    width = max(len(r) for r in rows)
    grid = np.zeros((len(rows), width), dtype=int)
    start = goal = None
    for r, line in enumerate(rows):
        for c, ch in enumerate(line.ljust(width, "#")):
            if ch == "#":
                grid[r, c] = 1
            elif ch == "S":
                start = (r, c)
            elif ch == "G":
                goal = (r, c)
    assert start is not None and goal is not None, "Maze needs an S and a G"
    return grid, start, goal

maze, maze_start, maze_goal = parse_grid(MAZE_TEXT)
room, room_start, room_goal = parse_grid(OPEN_TEXT)
print("maze:", maze.shape, "start", maze_start, "goal", maze_goal)
print("room:", room.shape, "start", room_start, "goal", room_goal)

### Drawing helper

You don't need to read this cell closely — it just turns a grid plus a search result into a picture.

In [ ]:
WALL = np.array([0.25, 0.25, 0.28])
FREE = np.array([1.0, 1.0, 1.0])
FRONT = np.array([1.0, 0.85, 0.25])

def render(grid, expanded=(), frontier=()):
    # Build an RGB image of the grid with expanded / frontier cells colored.
    img = np.where(grid[..., None] == 1, WALL, FREE).astype(float)
    n = max(len(expanded), 1)
    for i, (r, c) in enumerate(expanded):
        t = 0.25 + 0.6 * i / n                    # light blue -> darker blue over time
        img[r, c] = plt.cm.Blues(t)[:3]
    for (r, c) in frontier:
        img[r, c] = FRONT
    return img

def draw(grid, start, goal, result=None, title="", ax=None, show_frontier=True):
    if ax is None:
        _, ax = plt.subplots(figsize=(grid.shape[1] / 3, grid.shape[0] / 3))
    expanded = result["expanded"] if result else []
    frontier = result["frontier"] if (result and show_frontier) else []
    ax.imshow(render(grid, expanded, frontier), interpolation="nearest")
    if result and result["path"]:
        ys, xs = zip(*result["path"])
        ax.plot(xs, ys, color="crimson", lw=2.5)
    ax.plot(start[1], start[0], "o", color="limegreen", ms=10, mec="k")
    ax.plot(goal[1], goal[0], "*", color="red", ms=16, mec="k")
    ax.set_xticks([]); ax.set_yticks([])
    if result:
        cost = "none" if result["cost"] is None else f"{result['cost']:.2f}"
        title = f"{title}\nexpanded = {len(result['expanded'])},  path cost = {cost}"
    ax.set_title(title, fontsize=10)
    return ax

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
draw(maze, maze_start, maze_goal, title="MAZE (corridors)", ax=axes[0])
draw(room, room_start, room_goal, title="ROOM (open space with a wall)", ax=axes[1])
plt.tight_layout(); plt.show()

## Part 2 — Dijkstra recap

Dijkstra keeps a **priority queue** (the *frontier*) ordered by
$$g(n) = \text{cost of the best path found so far from the start to } n.$$

It repeatedly pops the node with the **smallest $g$**, marks it *expanded* (final), and relaxes its neighbors.

Because it only looks *backward* (how far have I come?), it has no idea where the goal is. It spreads out evenly in every direction — like a ripple in a pond — until the ripple happens to reach the goal.

Below is one search function we will reuse for **every** algorithm in this notebook. Read it carefully; it is short.
The only thing that changes between Dijkstra, A\*, Greedy and Weighted A\* is **how the priority is computed** (the highlighted line).

In [ ]:
SQRT2 = math.sqrt(2)

def neighbors(grid, node, diagonal=False):
    # Yield (neighbor, step_cost) for open cells next to node.
    r, c = node
    R, C = grid.shape
    steps = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if diagonal:
        steps += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    for dr, dc in steps:
        nr, nc = r + dr, c + dc
        if not (0 <= nr < R and 0 <= nc < C) or grid[nr, nc] == 1:
            continue
        if dr and dc and (grid[r, nc] == 1 or grid[nr, c] == 1):
            continue                     # no cutting corners through walls
        yield (nr, nc), (SQRT2 if dr and dc else 1.0)

def zero_h(a, b):
    return 0.0

def best_first_search(grid, start, goal, h=zero_h, w_g=1.0, w_h=1.0, diagonal=False, max_steps=None):
    '''
    Generic best-first search.  Priority of a node n is
            f(n) = w_g * g(n) + w_h * h(n, goal)

        Dijkstra        : h = 0                      (w_g=1)
        A*              : h = admissible heuristic   (w_g=1, w_h=1)
        Weighted A*     : w_h > 1
        Greedy best-1st : w_g = 0                    (ignore cost so far)
    '''
    tie = itertools.count()                       # breaks ties FIFO so heapq never compares nodes
    g = {start: 0.0}
    parent = {start: None}
    h0 = h(start, goal)
    frontier = [(w_g * 0.0 + w_h * h0, h0, next(tie), start)]
    expanded, expanded_set = [], set()
    pushed_at = {start: 0}                        # step at which a node first entered the frontier (for animation)

    while frontier:
        f, _, _, node = heapq.heappop(frontier)
        if node in expanded_set:                  # stale queue entry -- we already found a better one
            continue
        if max_steps is not None and len(expanded) >= max_steps:
            heapq.heappush(frontier, (f, _, next(tie), node))   # put it back: we stopped early
            break
        expanded.append(node); expanded_set.add(node)
        if node == goal:
            break
        for nb, step in neighbors(grid, node, diagonal):
            new_g = g[node] + step
            if nb not in g or new_g < g[nb] - 1e-12:
                g[nb] = new_g
                parent[nb] = node
                hn = h(nb, goal)
                priority = w_g * new_g + w_h * hn          # <<<<<<<< THE ONLY LINE THAT MATTERS
                heapq.heappush(frontier, (priority, hn, next(tie), nb))
                pushed_at.setdefault(nb, len(expanded))

    path, cost = [], None
    if goal in expanded_set:
        n = goal
        while n is not None:
            path.append(n); n = parent[n]
        path.reverse()
        cost = g[goal]
    open_nodes = {n for (_, _, _, n) in frontier} - expanded_set
    return {"path": path, "cost": cost, "expanded": expanded,
            "frontier": open_nodes, "pushed_at": pushed_at, "g": g}

def dijkstra(grid, start, goal, diagonal=False):
    return best_first_search(grid, start, goal, h=zero_h, diagonal=diagonal)

In [ ]:
d_maze = dijkstra(maze, maze_start, maze_goal)
d_room = dijkstra(room, room_start, room_goal)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
draw(maze, maze_start, maze_goal, d_maze, "Dijkstra on MAZE", ax=axes[0])
draw(room, room_start, room_goal, d_room, "Dijkstra on ROOM", ax=axes[1])
plt.tight_layout(); plt.show()

**Look at the ROOM picture.** Dijkstra found the shortest path, but it expanded almost every cell — including the whole left side of the room, *away* from the goal. It had no reason not to: a cell 10 steps to the left looks exactly as attractive as a cell 10 steps to the right.

> **Question to think about:** we *know* where the goal is. How could we tell the algorithm "cells that are closer to the goal are more promising"?

## Part 3 — Adding a heuristic: $f(n) = g(n) + h(n)$

A **heuristic** $h(n)$ is a cheap *estimate* of the remaining cost from $n$ to the goal. A\* orders the frontier by

$$
f(n) \;=\; \underbrace{g(n)}_{\text{cost so far (exact)}} \;+\; \underbrace{h(n)}_{\text{cost to go (estimate)}}
$$

so $f(n)$ estimates the cost of the *whole* path start → $n$ → goal. If $h(n) = 0$ for every node, $f = g$ and A\* **is** Dijkstra.

Two classic heuristics on a grid, for a cell $(r, c)$ and goal $(r_g, c_g)$:

| Heuristic | Formula | Intuition |
|---|---|---|
| **Manhattan** | $\lvert r - r_g\rvert + \lvert c - c_g\rvert$ | distance a taxi drives on a city grid (no diagonals) |
| **Euclidean** | $\sqrt{(r - r_g)^2 + (c - c_g)^2}$ | straight-line, "as the crow flies" |
| **Octile** (for 8-way moves) | $\max(\Delta r,\Delta c) + (\sqrt2 - 1)\min(\Delta r,\Delta c)$ | best you can do with diagonal steps of cost $\sqrt2$ |

Both ignore walls — that's what makes them cheap. They can only **underestimate** the true distance (walls only make paths longer), which turns out to be exactly the property we need.

In [ ]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def euclidean(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def octile(a, b):
    dr, dc = abs(a[0] - b[0]), abs(a[1] - b[1])
    return max(dr, dc) + (SQRT2 - 1) * min(dr, dc)

def astar(grid, start, goal, h=manhattan, diagonal=False):
    return best_first_search(grid, start, goal, h=h, diagonal=diagonal)

# Quick sanity check
print("manhattan((0,0),(3,4)) =", manhattan((0, 0), (3, 4)))   # 7
print("euclidean((0,0),(3,4)) =", euclidean((0, 0), (3, 4)))   # 5.0
print("octile   ((0,0),(3,4)) =", round(octile((0, 0), (3, 4)), 3))  # 4 + 0.414*3 = 5.243

### What does the heuristic "see"?

Here is $h(n)$ drawn as a heat map over the ROOM. Lighter = the heuristic thinks you are closer to the goal. Notice that it walks straight **through** the wall — the heuristic does not know walls exist.

In [ ]:
def heuristic_map(grid, goal, h):
    H = np.full(grid.shape, np.nan)
    for r in range(grid.shape[0]):
        for c in range(grid.shape[1]):
            if grid[r, c] == 0:
                H[r, c] = h((r, c), goal)
    return H

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for ax, (name, h) in zip(axes, [("Manhattan", manhattan), ("Euclidean", euclidean)]):
    H = heuristic_map(room, room_goal, h)
    ax.imshow(np.where(room[..., None] == 1, WALL, FREE))
    im = ax.imshow(H, cmap="magma_r", alpha=0.85)
    ax.plot(room_goal[1], room_goal[0], "*", color="red", ms=16, mec="k")
    ax.set_title(f"{name} distance to goal: h(n)")
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.025)
plt.tight_layout(); plt.show()

### Dijkstra vs. A\* side by side

In [ ]:
def compare(grid, start, goal, runs, cols=3, cell=0.33):
    # runs: list of (label, result). Draw them in a grid of subplots and return a summary table.
    rows = math.ceil(len(runs) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * grid.shape[1] * cell, rows * grid.shape[0] * cell + 0.8 * rows),
                             squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (label, res) in zip(axes.flat, runs):
        ax.axis("on")
        draw(grid, start, goal, res, label, ax=ax)
    plt.tight_layout(); plt.show()
    table = pd.DataFrame([{"algorithm": label,
                           "nodes expanded": len(res["expanded"]),
                           "path cost": None if res["cost"] is None else round(res["cost"], 3),
                           "path length (cells)": len(res["path"])} for label, res in runs])
    return table.set_index("algorithm")

runs_room = [
    ("Dijkstra (h = 0)",     dijkstra(room, room_start, room_goal)),
    ("A* + Euclidean",       astar(room, room_start, room_goal, euclidean)),
    ("A* + Manhattan",       astar(room, room_start, room_goal, manhattan)),
]
display(compare(room, room_start, room_goal, runs_room))

In [ ]:
runs_maze = [
    ("Dijkstra (h = 0)",     dijkstra(maze, maze_start, maze_goal)),
    ("A* + Euclidean",       astar(maze, maze_start, maze_goal, euclidean)),
    ("A* + Manhattan",       astar(maze, maze_start, maze_goal, manhattan)),
]
display(compare(maze, maze_start, maze_goal, runs_maze))

**What to notice**

* All three return a path with the **same cost** — A\* did not sacrifice optimality.
* In the open ROOM, A\* expands far fewer cells: the heuristic pulls the search toward the goal and around the wall.
* In the twisty MAZE the savings are smaller: the corridors force the search to wander away from the goal no matter what, and the heuristic (which ignores walls) is often a poor guess. **A heuristic is only as useful as it is informative.**
* Manhattan expands fewer cells than Euclidean. On a 4-connected grid both are admissible, but Manhattan is always $\ge$ Euclidean, so it is *closer to the truth*. We say Manhattan **dominates** Euclidean here.

### Watch it happen

The animation below replays the searches one expansion at a time. Dijkstra's frontier (yellow) is a growing ring; A\*'s frontier is a narrow wedge pointed at the goal. Use the play / step buttons under the animation.

In [ ]:
def animate_side_by_side(grid, start, goal, runs, n_frames=60, interval=120):
    # Replay several searches step by step in one animation.
    fig, axes = plt.subplots(1, len(runs), figsize=(len(runs) * grid.shape[1] * 0.3, grid.shape[0] * 0.3 + 1))
    axes = np.atleast_1d(axes)
    longest = max(len(r["expanded"]) for _, r in runs)
    steps = np.unique(np.linspace(0, longest, n_frames).astype(int))
    images = []
    for ax, (label, res) in zip(axes, runs):
        images.append(ax.imshow(render(grid), interpolation="nearest"))
        ax.plot(start[1], start[0], "o", color="limegreen", ms=8, mec="k")
        ax.plot(goal[1], goal[0], "*", color="red", ms=13, mec="k")
        ax.set_xticks([]); ax.set_yticks([])
    titles = [ax.set_title("", fontsize=10) for ax in axes]
    lines = [ax.plot([], [], color="crimson", lw=2.5)[0] for ax in axes]

    def frame(k):
        for img, ttl, line, (label, res) in zip(images, titles, lines, runs):
            done = min(k, len(res["expanded"]))
            exp = res["expanded"][:done]
            exp_set = set(exp)
            front = [n for n, t in res["pushed_at"].items() if t < done and n not in exp_set] if done < len(res["expanded"]) else res["frontier"]
            img.set_data(render(grid, exp, front))
            ttl.set_text(f"{label}\nexpanded {done}")
            if done == len(res["expanded"]) and res["path"]:
                ys, xs = zip(*res["path"]); line.set_data(xs, ys)
            else:
                line.set_data([], [])
        return images + lines + titles

    anim = animation.FuncAnimation(fig, frame, frames=list(steps) + [longest] * 8, interval=interval, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())

animate_side_by_side(room, room_start, room_goal,
                     [("Dijkstra", runs_room[0][1]), ("A* + Manhattan", runs_room[2][1])])

## Part 4 — What makes a heuristic "good"?

A\* returns an **optimal** path as long as the heuristic is

* **Admissible:** it never overestimates. For every node, $h(n) \le h^*(n)$, the true cheapest cost from $n$ to the goal.
* **Consistent** (a.k.a. monotone): for every edge $n \to n'$ with cost $c(n,n')$,
  $$h(n) \;\le\; c(n, n') + h(n').$$
  This is the triangle inequality. It guarantees that once a node is expanded, we already have its best path — just like in Dijkstra. Every consistent heuristic is admissible.

Among admissible heuristics, **bigger is better**: the closer $h$ is to $h^*$, the fewer nodes A\* expands. The two extremes:

| $h(n)$ | Behavior |
|---|---|
| $0$ | Dijkstra: optimal, explores everything |
| $h^*(n)$ (perfect) | walks straight down the optimal path, expands only those cells |
| $> h^*(n)$ somewhere | **not admissible**: often faster, but may return a *worse* path |

Let's test those boundaries.

### Experiment 4a — Ignore $g$ entirely: Greedy Best-First Search

What if we *only* look at the heuristic, $f(n) = h(n)$? The search rushes toward whatever cell *looks* closest, never asking how much it has already spent to get there.

In [ ]:
def greedy(grid, start, goal, h=manhattan, diagonal=False):
    return best_first_search(grid, start, goal, h=h, w_g=0.0, diagonal=diagonal)

runs = [
    ("Dijkstra",             dijkstra(room, room_start, room_goal)),
    ("A* + Manhattan",       astar(room, room_start, room_goal, manhattan)),
    ("Greedy (f = h only)",  greedy(room, room_start, room_goal, manhattan)),
]
display(compare(room, room_start, room_goal, runs))

runs = [
    ("Dijkstra",             dijkstra(maze, maze_start, maze_goal)),
    ("A* + Manhattan",       astar(maze, maze_start, maze_goal, manhattan)),
    ("Greedy (f = h only)",  greedy(maze, maze_start, maze_goal, manhattan)),
]
display(compare(maze, maze_start, maze_goal, runs))

In the ROOM, Greedy got lucky and still found a shortest path. In the MAZE it did not: it charged down a corridor that pointed toward the goal and committed to a route that costs noticeably more (compare the path costs in the second table). Greedy can be very fast, but it has **no optimality guarantee**. It is the opposite extreme from Dijkstra: Dijkstra uses only $g$, Greedy uses only $h$. A\* balances the two.

### Experiment 4b — Inflate the heuristic: Weighted A\*

Weighted A\* uses $f(n) = g(n) + w \cdot h(n)$ with $w > 1$. For $w>1$ the heuristic is no longer admissible. The search gets greedier as $w$ grows. (A useful fact: the returned path is guaranteed to cost at most $w$ times the optimum.)

In [ ]:
weights = [0, 0.5, 1, 1.5, 2, 3, 5, 10]
rows = []
for name, (G, S, T) in {"ROOM": (room, room_start, room_goal), "MAZE": (maze, maze_start, maze_goal)}.items():
    opt = dijkstra(G, S, T)["cost"]
    for w in weights:
        res = best_first_search(G, S, T, h=manhattan, w_h=w)
        rows.append({"map": name, "w": w, "expanded": len(res["expanded"]),
                     "cost": res["cost"], "cost / optimal": round(res["cost"] / opt, 3)})
wa = pd.DataFrame(rows)
display(wa.pivot(index="w", columns="map", values=["expanded", "cost / optimal"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name, sub in wa.groupby("map"):
    axes[0].plot(sub["w"], sub["expanded"], "o-", label=name)
    axes[1].plot(sub["w"], sub["cost / optimal"], "o-", label=name)
axes[0].set(xlabel="heuristic weight w", ylabel="nodes expanded", title="Work done")
axes[1].set(xlabel="heuristic weight w", ylabel="path cost / optimal", title="Path quality")
axes[1].axhline(1, color="gray", ls="--", lw=1)
for ax in axes:
    ax.axvline(1, color="green", ls=":", lw=1); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

* $w = 0$ is Dijkstra, $w = 1$ is plain A\* (green dotted line).
* For $w \le 1$ the cost ratio stays at 1.0 — still optimal.
* For $w > 1$ the work usually drops but the path can get longer. That is the **speed vs. optimality trade-off**, and it is controlled entirely by how much we trust $h$.

### Experiment 4c — The right heuristic depends on how you can move

Now allow **diagonal moves** (cost $\sqrt2$). Suddenly Manhattan **overestimates**: from $(0,0)$ to $(1,1)$ Manhattan says 2, but one diagonal step costs only $\sqrt2 \approx 1.41$. So Manhattan is *no longer admissible*. The heuristic that matches 8-way movement exactly in empty space is **octile** distance. Euclidean is still admissible but weaker.

In [ ]:
runs = [
    ("8-way Dijkstra",          dijkstra(room, room_start, room_goal, diagonal=True)),
    ("8-way A* + Euclidean",    astar(room, room_start, room_goal, euclidean, diagonal=True)),
    ("8-way A* + Octile",       astar(room, room_start, room_goal, octile, diagonal=True)),
    ("8-way A* + Manhattan (!)", astar(room, room_start, room_goal, manhattan, diagonal=True)),
]
display(compare(room, room_start, room_goal, runs, cols=2))

On this particular map all four costs match: Manhattan got lucky. One lucky map proves nothing, so let's test it on 40 random maps with diagonal moves allowed:

In [ ]:
rng = np.random.default_rng(1)
rows = []
for i in range(40):
    G = (rng.random((35, 35)) < 0.28).astype(int)
    G[0, :] = G[-1, :] = G[:, 0] = G[:, -1] = 1
    S, T = (1, 1), (33, 33); G[S] = G[T] = 0
    best = dijkstra(G, S, T, diagonal=True)["cost"]
    if best is None:
        continue                                  # skip unsolvable maps
    for name, h in [("Euclidean", euclidean), ("Octile", octile), ("Manhattan", manhattan)]:
        c = astar(G, S, T, h, diagonal=True)["cost"]
        rows.append({"heuristic": name, "optimal?": abs(c - best) < 1e-9, "cost / optimal": c / best})
res8 = pd.DataFrame(rows).groupby("heuristic", sort=False).agg(
    maps=("optimal?", "size"), pct_optimal=("optimal?", lambda s: 100 * s.mean()),
    worst_cost_ratio=("cost / optimal", "max"))
display(res8.round(3))

Manhattan, which overestimates once diagonals are allowed, returns a longer-than-necessary path on many of these maps, and it gives no warning when it does. Octile and Euclidean are always optimal. **Always check admissibility against the movement rules you actually use.**

| Movement | Admissible heuristics | Best (tightest) choice |
|---|---|---|
| 4-way, cost 1 | Manhattan, Euclidean, 0 | **Manhattan** |
| 8-way, straight 1 / diagonal $\sqrt2$ | Octile, Euclidean, 0 | **Octile** |
| any-angle / continuous | Euclidean, 0 | **Euclidean** |

## Part 5 — Many random mazes

One map can be misleading. Let's generate many random grids (random walls, a fixed fraction of the cells blocked), run each algorithm, and average the results. Change `N_MAPS`, `SIZE`, or `WALL_DENSITY` and re-run.

In [ ]:
def random_grid(rows, cols, density, rng):
    # Random walls with a solid border. Start = top-left area, goal = bottom-right area.
    while True:
        g = (rng.random((rows, cols)) < density).astype(int)
        g[0, :] = g[-1, :] = g[:, 0] = g[:, -1] = 1
        s, t = (1, 1), (rows - 2, cols - 2)
        g[s] = g[t] = 0
        if dijkstra(g, s, t)["cost"] is not None:   # keep only solvable mazes
            return g, s, t

N_MAPS, SIZE, WALL_DENSITY = 40, 35, 0.28
rng = np.random.default_rng(0)
algos = {
    "Dijkstra":       lambda G, S, T: dijkstra(G, S, T),
    "A* Euclidean":   lambda G, S, T: astar(G, S, T, euclidean),
    "A* Manhattan":   lambda G, S, T: astar(G, S, T, manhattan),
    "Greedy":         lambda G, S, T: greedy(G, S, T, manhattan),
}
records = []
for i in range(N_MAPS):
    G, S, T = random_grid(SIZE, SIZE, WALL_DENSITY, rng)
    opt = None
    for name, fn in algos.items():
        res = fn(G, S, T)
        if name == "Dijkstra":
            opt = res["cost"]
        records.append({"map": i, "algorithm": name, "expanded": len(res["expanded"]),
                        "optimal?": abs(res["cost"] - opt) < 1e-9})
df = pd.DataFrame(records)
summary = df.groupby("algorithm", sort=False).agg(mean_expanded=("expanded", "mean"),
                                                  pct_optimal=("optimal?", lambda s: 100 * s.mean()))
display(summary.round(1))

ax = summary["mean_expanded"].plot.bar(figsize=(7, 3.5), color=["#4c72b0", "#55a868", "#2a9d8f", "#c44e52"], rot=0)
ax.set_ylabel("mean nodes expanded"); ax.set_title(f"{N_MAPS} random {SIZE}x{SIZE} grids, {int(WALL_DENSITY*100)}% walls")
plt.tight_layout(); plt.show()

# Show one example map
G, S, T = random_grid(SIZE, SIZE, WALL_DENSITY, np.random.default_rng(7))
display(compare(G, S, T, [(n, fn(G, S, T)) for n, fn in algos.items()], cols=4, cell=0.22))

## Part 6 — Interactive explorer

Pick a map, an algorithm, and a step number to see the state of the search at that moment (expanded set, frontier, and — once finished — the path). Also try the **"f-values"** checkbox to print $g$, $h$, and $f$ for the next few nodes the algorithm is about to pop.

*(If the widgets don't appear, run `!pip install ipywidgets` and re-run this cell. In Colab they work out of the box.)*

In [ ]:
import ipywidgets as widgets

MAPS = {"ROOM": (room, room_start, room_goal), "MAZE": (maze, maze_start, maze_goal)}
ALGS = {
    "Dijkstra (h=0)":   dict(h=zero_h),
    "A* Euclidean":     dict(h=euclidean),
    "A* Manhattan":     dict(h=manhattan),
    "Weighted A* (w=3)": dict(h=manhattan, w_h=3.0),
    "Greedy (f=h)":     dict(h=manhattan, w_g=0.0),
}

def explore(map_name="ROOM", algorithm="A* Manhattan", step=40, show_f=False):
    G, S, T = MAPS[map_name]
    opts = ALGS[algorithm]
    total = len(best_first_search(G, S, T, **opts)["expanded"])
    res = best_first_search(G, S, T, max_steps=step, **opts)    # stop after `step` expansions
    done = len(res["expanded"])
    draw(G, S, T, res, f"{algorithm} on {map_name} -- step {done}/{total}")
    plt.show()
    if show_f and res["frontier"]:
        h, wg, wh = opts.get("h", zero_h), opts.get("w_g", 1.0), opts.get("w_h", 1.0)
        rows = [{"node": n, "g": round(res["g"][n], 2), "h": round(h(n, T), 2),
                 "priority f": round(wg * res["g"][n] + wh * h(n, T), 2)} for n in res["frontier"]]
        print("Next nodes to be popped (lowest priority first):")
        display(pd.DataFrame(rows).sort_values(["priority f", "h"]).head(8).reset_index(drop=True))

widgets.interact(explore,
                 map_name=widgets.Dropdown(options=list(MAPS), value="ROOM", description="map"),
                 algorithm=widgets.Dropdown(options=list(ALGS), value="A* Manhattan", description="algorithm"),
                 step=widgets.IntSlider(min=0, max=300, step=1, value=40, description="step",
                                        continuous_update=False),
                 show_f=widgets.Checkbox(value=False, description="f-values"));

## Part 7 — Exercises

1. **Trace by hand.** On the tiny grid below (start at row 2, column 2; goal at row 3, column 7), write out the order in which Dijkstra and A\* (Manhattan) expand cells, with their $g$, $h$, and $f$ values. Then run the cell to check yourself.
2. **Prove it.** Show that Manhattan distance is *consistent* on a 4-connected grid with unit costs. (Hint: one step changes $\lvert r - r_g\rvert + \lvert c - c_g\rvert$ by exactly 1.)
3. **Break it.** Write a heuristic `my_h` that is *not* admissible and find a map (edit `OPEN_TEXT` or draw your own) where A\* with `my_h` returns a path that is **longer** than Dijkstra's.
4. **Dominance.** Try `h = lambda a, b: max(abs(a[0]-b[0]), abs(a[1]-b[1]))` (Chebyshev distance) on the 4-connected ROOM. Is it admissible? Does it expand more or fewer cells than Manhattan? Why?
5. **Terrain.** Real maps have mud, roads, and water. Modify `neighbors` so that some cells cost 5 to enter. Is Manhattan distance still admissible? What if some cells cost **0.5**?
6. **Design question.** A\* stores every frontier node. For a $10{,}000 \times 10{,}000$ grid, why could that be a problem, and what would you trade away to fix it?

In [ ]:
TINY = '''
#########
#...#...#
#.S.....#
#...#..G#
#########
'''
tiny, ts, tg = parse_grid(TINY)

for label, h in [("Dijkstra", zero_h), ("A* Manhattan", manhattan)]:
    res = best_first_search(tiny, ts, tg, h=h)
    g = res["g"]
    print(f"{label}: expansion order (node: g, h, f)")
    for n in res["expanded"]:
        print(f"   {n}: g={g[n]:.0f}, h={h(n, tg):.0f}, f={g[n] + h(n, tg):.0f}")
    print()

display(compare(tiny, ts, tg, [("Dijkstra", dijkstra(tiny, ts, tg)), ("A* Manhattan", astar(tiny, ts, tg))], cols=2, cell=0.7))

In [ ]:
# Exercise 3 starter: define an inadmissible heuristic and test it
def my_h(a, b):
    return 5 * manhattan(a, b)      # <-- change me

my_map, ms, mg = parse_grid(OPEN_TEXT)  # or parse_grid(YOUR_TEXT)
display(compare(my_map, ms, mg, [("Dijkstra", dijkstra(my_map, ms, mg)),
                                 ("A* with my_h", astar(my_map, ms, mg, my_h))], cols=2))

## Summary

* **Dijkstra** orders the frontier by $g(n)$ — what it *has* spent. Optimal, but blind to where the goal is.
* **A\*** orders by $f(n) = g(n) + h(n)$ — what it has spent plus an *estimate* of what is left. Same code, one extra term.
* If $h$ is **admissible** (never overestimates) and **consistent**, A\* is still optimal, and it never expands more nodes than Dijkstra.
* A **tighter** admissible heuristic (closer to the true distance) means fewer expansions: on a 4-way grid, Manhattan beats Euclidean; with diagonals, use octile.
* Ignoring $g$ (Greedy) or inflating $h$ (Weighted A\*) trades optimality for speed.

**Further reading:** Amit Patel's *Introduction to A\** (Red Blob Games), and Russell & Norvig, *Artificial Intelligence: A Modern Approach*, Ch. 3 (informed search).